In [0]:
# Step 00: Reset the Module 1 demo

# Stop any active streaming queries
for q in spark.streams.active:
    q.stop()

catalog = "retail"
schema = "sales"
volume = "autoloader_demo"

base = f"/Volumes/{catalog}/{schema}/{volume}/auto_loader"

landing = f"{base}/landing"
schema_loc = f"{base}/_schema"
checkpoint = f"{base}/_checkpoint"

table_name = f"{catalog}.{schema}.orders_bronze"

# Drop the old table
spark.sql(f"DROP TABLE IF EXISTS {table_name}")

# Remove old files, schema metadata and checkpoint
dbutils.fs.rm(base, True)

# Recreate the landing folder
dbutils.fs.mkdirs(landing)

print("Module 1 demo reset completed.")
print(f"Landing: {landing}")
print(f"Schema location: {schema_loc}")
print(f"Checkpoint: {checkpoint}")
print(f"Table: {table_name}")

In [0]:
# Databricks Auto Loader - Incremental Ingestion

# Step 01: Setup paths

catalog = "retail"
schema = "sales"
volume = "autoloader_demo"

base = f"/Volumes/{catalog}/{schema}/{volume}/auto_loader"

landing = f"{base}/landing"
schema_loc = f"{base}/_schema"
checkpoint = f"{base}/_checkpoint"

table_name = f"{catalog}.{schema}.orders_bronze"

# Create landing folder
dbutils.fs.mkdirs(landing)

In [0]:
# ============================================================
#  Step 02 — Create initial sample JSON files
# ============================================================

# Create the first JSON file

dbutils.fs.put(
    f"{landing}/orders_01.json",
    '{"order_id": 1, "customer": "Aditi", "amount": 250}\n'
    '{"order_id": 2, "customer": "Rahul", "amount": 120}\n',
    True
)

# Create the second JSON file

dbutils.fs.put(
    f"{landing}/orders_02.json",
    '{"order_id": 3, "customer": "Meera", "amount": 890}\n',
    True
)

In [0]:

# ============================================================
# Step 03 - Check the landing folder
# ============================================================

display(dbutils.fs.ls(landing))


In [0]:
# ============================================================
# Step 04 — Define Auto Loader
# ============================================================

orders_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_loc)
    .load(landing)
)

In [0]:

# ============================================================
# Step 05 — Start the stream
# ============================================================

query = (
    orders_stream.writeStream
    .option("checkpointLocation", checkpoint)
    .trigger(availableNow=True)
    .toTable(table_name)
)


In [0]:

# ============================================================
# Step 06 — Check the Bronze table
# ============================================================

display(
    spark.table(table_name)
)


In [0]:
# ============================================================
# Step 07 — Add a new file
# ============================================================

dbutils.fs.put(
    f"{landing}/orders_03.json",
    '{"order_id": 4, "customer": "Sanjay", "amount": 410}\n',
    True
)

In [0]:

# ============================================================
# Step 08 — Read the landing folder again with Auto Loader
# ============================================================

orders_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_loc)
    .load(landing)
)

In [0]:

# ============================================================
# Step 09 — Use the SAME checkpoint
# ============================================================
query = (
    orders_stream.writeStream
    .option("checkpointLocation", checkpoint)
    .trigger(availableNow=True)
    .toTable(table_name)
)

In [0]:
#Step 10 — Verify the final table


display(
    spark.table(table_name)
    .orderBy("order_id")
)